# ============================================
# Cell 1: Import libraries (if not already imported)
# ============================================

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.basemap import Basemap

# ============================================
# Cell 2: Define function to process ozone data for new cases
# ============================================

In [ ]:
def process_o3_data_newcase(file_pattern, prefix):
    """
    Process ozone data for a new case.
    Reads multiple nc files matching file_pattern, computes zonal mean,
    selects 60°–90° latitude (weighted average) and computes partial ozone columns for 30–70 and 30–100 hPa.
    Returns a dictionary with keys '{prefix}_30_70' and '{prefix}_30_100'.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    o3 = ds['O3'].mean(dim='lon')
    # Compute 30-70 hPa partial column
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    data_30_70 = o3_30_70.sel(lat=slice(60,90))
    weights = np.cos(np.deg2rad(data_30_70.lat))
    avg_30_70 = data_30_70.weighted(weights).mean(dim='lat')
    # Compute 30-100 hPa partial column
    o3_30_100 = calc_parc_o3(o3, 30, 100)
    data_30_100 = o3_30_100.sel(lat=slice(60,90))
    weights = np.cos(np.deg2rad(data_30_100.lat))
    avg_30_100 = data_30_100.weighted(weights).mean(dim='lat')
    return {f'{prefix}_30_70': avg_30_70, f'{prefix}_30_100': avg_30_100}


# ============================================
# Cell 3: Define plotting function for ozone evolution with case‐specific reference
# ============================================

In [ ]:
def plot_o3_evolution_newcase_with_ref(var_F, var_M, var_clim, pressure_range='30-70', case_id=''):
    """
    Plot ozone evolution (Feb and Mar) for a new case with overlaid reference curves.
    The reference curves are extracted from var_clim based on the case_id year.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    
    if var_F is not None:
        x_F = np.arange(31, 31+len(var_F.time))
        ax.plot(x_F, var_F.mean(dim='member'), color='navy', linewidth=3, label='Feb (new)')
        ax.fill_between(x_F, var_F.min(dim='member'), var_F.max(dim='member'), color='navy', alpha=0.3)
    
    if var_M is not None:
        x_M = np.arange(59, 59+len(var_M.time))
        ax.plot(x_M, var_M.mean(dim='member'), color='crimson', linewidth=3, label='Mar (new)')
        ax.fill_between(x_M, var_M.min(dim='member'), var_M.max(dim='member'), color='crimson', alpha=0.3)
    
    try:
        case_year = int(case_id)
    except:
        case_year = None
    ref_curve = None
    ref_clim = None
    if case_year is not None:
        temp_sel = var_clim.sel(time=var_clim.time.dt.month.isin([1,2,3,4,5,6]))
        ref_curve = temp_sel.sel(time=temp_sel.time.dt.year==case_year)
        ref_clim = temp_sel.groupby("time.dayofyear").mean()
    
    if ref_curve is not None:
        x_ref = np.arange(len(ref_curve))
        ax.plot(x_ref, ref_curve, color='black', linestyle='--', linewidth=2, label=f'Ref ({case_year})')
    if ref_clim is not None:
        x_ref_clim = np.arange(len(ref_clim))
        ax.plot(x_ref_clim, ref_clim, color='black', linestyle=':', linewidth=2, label='Clim (all)')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Partial Ozone Column ({pressure_range} hPa) (DU)')
    ax.legend()
    
    save_pdf = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_{case_id}_{pressure_range.replace("-","to")}_withRef.pdf'
    save_png = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_{case_id}_{pressure_range.replace("-","to")}_withRef.png'
    plt.savefig(save_pdf)
    plt.savefig(save_png)
    plt.close(fig)
    print(f"Saved ozone evolution plot with reference for case {case_id} ({pressure_range})")
    return fig, ax

# ============================================
# Cell 4: Define functions for extended temperature and wind evolution with reference
# ============================================

In [ ]:
def plot_temp_evolution_newcase_with_ref(var_base, var_base_clim, var_F, var_M, case_id, save_path, plev=5000, lat_range=(60,90)):
    """
    Plot temperature evolution for a new case (Feb and Mar) with overlaid reference curves.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    try:
        case_year = int(case_id)
    except:
        case_year = None
    ref_curve = None
    ref_clim = None
    if case_year is not None:
        temp_sel = var_base.sel(time=var_base.time.dt.month.isin([1,2,3,4,5,6]))
        ref_curve = temp_sel.sel(time=temp_sel.time.dt.year==case_year)
        ref_clim = temp_sel.groupby("time.dayofyear").mean()
    if ref_curve is not None:
        x_ref = np.arange(len(ref_curve))
        ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if ref_clim is not None:
        x_ref_clim = np.arange(len(ref_clim))
        ax.plot(x_ref_clim, ref_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    if var_F is not None:
        x_F = np.arange(31, 31+len(var_F.time))
        ax.plot(x_F, var_F.mean(dim='member'), color='royalblue', linewidth=3, label='Feb (extra)')
        ax.fill_between(x_F, var_F.min(dim='member'), var_F.max(dim='member'), color='royalblue', alpha=0.3)
    if var_M is not None:
        x_M = np.arange(59, 59+len(var_M.time))
        ax.plot(x_M, var_M.mean(dim='member'), color='hotpink', linewidth=3, label='Mar (extra)')
        ax.fill_between(x_M, var_M.min(dim='member'), var_M.max(dim='member'), color='hotpink', alpha=0.3)
    
    # Add chlorine activation threshold line (e.g. 197K)
    ax.axhline(y=197, color='royalblue', linestyle='--')
    ax.text(5, 194, 'Cl activation threshold', fontsize=20, color='royalblue')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev/100:.0f} hPa (K)')
    ax.legend()
    
    plt.savefig(save_path + '.pdf')
    plt.savefig(save_path + '.png')
    plt.close(fig)
    print(f"Saved temperature evolution plot with reference for case {case_id}")
    return fig, ax

def plot_wind_evolution_newcase_with_ref(var_base, var_base_clim, var_F, var_M, case_id, save_path, plev=1000, lat=60):
    """
    Plot zonal wind evolution for a new case (Feb and Mar) with overlaid reference curves.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    try:
        case_year = int(case_id)
    except:
        case_year = None
    ref_curve = None
    ref_clim = None
    if case_year is not None:
        wind_sel = var_base.sel(time=var_base.time.dt.month.isin([1,2,3,4,5,6]))
        ref_curve = wind_sel.sel(time=wind_sel.time.dt.year==case_year)
        ref_clim = wind_sel.groupby("time.dayofyear").mean()
    if ref_curve is not None:
        x_ref = np.arange(len(ref_curve))
        ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if ref_clim is not None:
        x_ref_clim = np.arange(len(ref_clim))
        ax.plot(x_ref_clim, ref_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    if var_F is not None:
        x_F = np.arange(31, 31+len(var_F.time))
        ax.plot(x_F, var_F.mean(dim='member'), color='royalblue', linewidth=3, label='Feb (extra)')
        ax.fill_between(x_F, var_F.min(dim='member'), var_F.max(dim='member'), color='royalblue', alpha=0.3)
    if var_M is not None:
        x_M = np.arange(59, 59+len(var_M.time))
        ax.plot(x_M, var_M.mean(dim='member'), color='hotpink', linewidth=3, label='Mar (extra)')
        ax.fill_between(x_M, var_M.min(dim='member'), var_M.max(dim='member'), color='hotpink', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Zonal Mean Zonal Wind, {lat}°N, {plev/100:.0f} hPa (m/s)')
    ax.legend()
    
    plt.savefig(save_path + '.pdf')
    plt.savefig(save_path + '.png')
    plt.close(fig)
    print(f"Saved wind evolution plot with reference for case {case_id}")
    return fig, ax

# ============================================
# Cell 5: Loop over new cases and generate extended plots
# ============================================

In [ ]:
NEW_CASES = ['0013', '0014', '0019', '0003']

for case in NEW_CASES:
    case_base_dir = f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{case}"
    
    # ----- Ozone Data Processing -----
    case_data = {}
    feb_dir = os.path.join(case_base_dir, "Feb")
    mar_dir = os.path.join(case_base_dir, "Mar")
    
    if os.path.isdir(feb_dir):
        feb_pattern = os.path.join(feb_dir, f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
        try:
            data_F = process_o3_data_newcase(feb_pattern, prefix="F")
            case_data['F_30_70'] = data_F["F_30_70"]
            case_data['F_30_100'] = data_F["F_30_100"]
        except Exception as e:
            print(f"Error processing ozone Feb data for case {case}: {e}")
            case_data['F_30_70'] = None
            case_data['F_30_100'] = None
    else:
        print(f"Feb directory not found for case {case} (ozone)")
        case_data['F_30_70'] = None
        case_data['F_30_100'] = None

    if os.path.isdir(mar_dir):
        mar_pattern = os.path.join(mar_dir, f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
        try:
            data_M = process_o3_data_newcase(mar_pattern, prefix="M")
            case_data['M_30_70'] = data_M["M_30_70"]
            case_data['M_30_100'] = data_M["M_30_100"]
        except Exception as e:
            print(f"Error processing ozone Mar data for case {case}: {e}")
            case_data['M_30_70'] = None
            case_data['M_30_100'] = None
    else:
        print(f"Mar directory not found for case {case} (ozone)")
        case_data['M_30_70'] = None
        case_data['M_30_100'] = None

    # Plot ozone evolution for both pressure ranges
    for pressure in ['30-70', '30-100']:
        key_F = 'F_30_70' if pressure=='30-70' else 'F_30_100'
        key_M = 'M_30_70' if pressure=='30-70' else 'M_30_100'
        var_F = case_data.get(key_F, None)
        var_M = case_data.get(key_M, None)
        if (var_F is None) and (var_M is None):
            print(f"No ozone data for pressure {pressure} in case {case} – skipping ozone plot.")
        else:
            # Use merged file to extract common climatology for reference
            merged_file = os.path.join(case_base_dir, "BWCN.e122.f19_g16.merged.nc")
            ds_merged = xr.open_dataset(merged_file)
            o3_merged = ds_merged['O3'].mean(dim='lon')
            o3_sel = o3_merged.sel(time=o3_merged.time.dt.month.isin([1,2,3,4,5,6]))
            clim_common = o3_sel.groupby("time.dayofyear").mean()
            plot_o3_evolution_newcase_with_ref(var_F, var_M, clim_common, pressure_range=pressure, case_id=case)
    
    # ----- Temperature Data Processing -----
    temp_base_file = os.path.join(case_base_dir, "BWCN.e122.f19_g16.merged.nc")
    try:
        var_temp_base, var_temp_clim = process_temp_base_data(temp_base_file, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing base temperature for case {case}: {e}")
        var_temp_base, var_temp_clim = None, None
    feb_temp_pattern = os.path.join(case_base_dir, "Feb", f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
    try:
        var_temp_F = process_temp_data(feb_temp_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing Feb temperature for case {case}: {e}")
        var_temp_F = None
    mar_temp_pattern = os.path.join(case_base_dir, "Mar", f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
    try:
        var_temp_M = process_temp_data(mar_temp_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing Mar temperature for case {case}: {e}")
        var_temp_M = None
        
    temp_save_path = f'/home/weiji/restart_exam/plots/T_evolution_min_restart_{case}_withRef'
    if var_temp_base is not None and var_temp_clim is not None:
        plot_temp_evolution_newcase_with_ref(var_temp_base, var_temp_clim, var_temp_F, var_temp_M, case, temp_save_path, plev=5000, lat_range=(60,90))
    
    # ----- Wind Data Processing -----
    def process_base_data(file_path, lat=60, plev=1000, months=[1,2,3,4,5,6]):
        ds = xr.open_dataset(file_path)
        wind = ds['U'].sel(plev=plev)
        wind = wind.sel(time=ds.time.dt.month.isin(months))
        wind = wind.mean(dim='lon')
        wind_sel = wind.sel(lat=lat, method='nearest')
        base = wind_sel.sel(time=wind_sel.time.dt.year==int(case))
        clim = wind_sel.groupby("time.dayofyear").mean()
        return base, clim

    wind_base_file = os.path.join(case_base_dir, "BWCN.e122.f19_g16.merged.nc")
    try:
        var_wind_base, var_wind_clim = process_base_data(wind_base_file, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing base wind for case {case}: {e}")
        var_wind_base, var_wind_clim = None, None
    feb_wind_pattern = os.path.join(case_base_dir, "Feb", f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
    try:
        var_wind_F = process_waccm_data(feb_wind_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing Feb wind for case {case}: {e}")
        var_wind_F = None
    mar_wind_pattern = os.path.join(case_base_dir, "Mar", f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
    try:
        var_wind_M = process_waccm_data(mar_wind_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing Mar wind for case {case}: {e}")
        var_wind_M = None
        
    wind_save_path = f'/home/weiji/restart_exam/plots/U_evolution_min_restart_{case}_withRef'
    if var_wind_base is not None and var_wind_clim is not None:
        plot_wind_evolution_newcase_with_ref(var_wind_base, var_wind_clim, var_wind_F, var_wind_M, case, wind_save_path, plev=1000, lat=60)

plt.close('all')